# Bengaluru Air Pollution — Data Exploration & Visualization
**Original dataset:** https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india

**Analyzed public extract:** 731 daily Bengaluru records, 2015–2016.

Run all cells in order. Only observed values enter the reported correlations; missing AQI is never filled.

## 1. Load historical public CSV and preserve raw values

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis import DATA_URL, load_data, audit_and_clean, summarize, make_plots, export_evidence, CORE, POLLUTANTS

source = Path("data/city_day.csv")
source.parent.mkdir(exist_ok=True)
if not source.exists():
    urllib.request.urlretrieve(DATA_URL, source)
raw = load_data(source)
print("Raw Bengaluru records:", len(raw)); display(raw.head(8))

## 2. Validate types, missingness, duplicates, invalid entries and outliers
**Policy:** Missing and outlier readings are transparently flagged; non-AQI pollutant median imputations are optional and excluded from primary analysis.

In [ ]:
df, quality = audit_and_clean(raw)
display(pd.DataFrame({"field":list(quality["missing_before"]),"missing":list(quality["missing_before"].values())}))
print("Duplicates:", quality["duplicates_city_date"], "Invalid dates:", quality["invalid_dates"])
print("IQR outliers:", quality["iqr_outliers"])

## 3. Before / after data cleaning

In [ ]:
cols = ["Date","PM2.5","PM10","NO2","CO","AQI","AQI_Bucket"]
print("BEFORE")
display(raw.head(8)[cols])
print("AFTER (observed values, no silent filling)")
display(df.head(8)[cols+["PM2.5_imputed","PM2.5_was_imputed","PM10_imputed","PM10_was_imputed"]])

## 4. Descriptive statistics and univariate analysis (objectives 2–3)

In [ ]:
display(df[CORE].describe().T)
print(df.AQI_Bucket.value_counts(dropna=False))
df.AQI.dropna().hist(bins=22,figsize=(9,4))
plt.title("Bengaluru AQI histogram");plt.xlabel("AQI");plt.ylabel("Days");plt.show()

**Interpretation:** A histogram is appropriate because AQI is numeric. The observed median is near 101, with a high tail; unusual days are retained because they may reflect genuine pollution episodes.

## 5. Bivariate comparisons (objectives 4–9)

In [ ]:
summary = summarize(df, quality)
print("Year-wise AQI means:", {k:v["mean"] for k,v in summary["year"].items()})
print("Season-wise AQI means:", {k:v["mean"] for k,v in summary["season"].items()})
print("Day-type means:", {k:v["mean"] for k,v in summary["daytype"].items()})
print("Pollutant correlation (with paired-day n):", summary["aqi_correlations"])

In [ ]:
for col in ["PM2.5","PM10"]:
    a = df[[col,"AQI"]].dropna()
    fig,ax=plt.subplots(figsize=(7,4));ax.scatter(a[col],a.AQI,alpha=.35)
    ax.set(xlabel=col,ylabel="AQI",title=f"{col} vs AQI • n={len(a)} • r={a[col].corr(a.AQI):.3f}")
    plt.show()

**Interpretation:** Scatterplots show the actual day-wise paired observations and reveal exceptions that a correlation coefficient hides. PM10–AQI shows a stronger positive Pearson association than PM2.5–AQI in this specific extract; the different pair counts and missingness matter.

## 6. Multivariate correlation analysis (objective 10)

In [ ]:
corr = df[CORE].corr()
display(corr.round(3))
fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(corr,vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(len(CORE)), CORE, rotation=45, ha="right")
ax.set_yticks(range(len(CORE)), CORE)
for i in range(len(CORE)):
    for j in range(len(CORE)):
        ax.text(j,i,f"{corr.iloc[i,j]:.2f}",va="center",ha="center")
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()

**Interpretation:** The heatmap summarizes several pairwise linear relationships. Positive PM2.5–NO₂ and PM10–NO₂ correlations do not establish shared sources without external emission/meteorological data.

## 7. Export all 11 charts, clean tables and audit files

In [ ]:
import analysis
from pathlib import Path
import json
analysis.CHARTS=Path("outputs/charts")
analysis.CHARTS.mkdir(parents=True,exist_ok=True)
analysis.make_plots(df)
out=Path("outputs");out.mkdir(exist_ok=True)
analysis.export_evidence(raw,df,quality,summary,out)
print("Wrote",len(list(analysis.CHARTS.glob("*.png"))),"charts and audit files")
print(json.dumps({"rows":summary["rows"],"aqi_stats":summary["aqi"]},indent=2))

## Limitations, future scope and source links
- Daily city-level records cannot support station or hourly claims.
- Significant missing data; no imputation of AQI; report pairwise sample sizes.
- Correlation is not causation, and AQI mechanically depends on pollutants.
- Add meteorology, traffic, higher-resolution monitoring and missing-data sensitivity checks.
- Original dataset: https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india
- GitHub code: https://github.com/Abhigyan-Shekhar/bengaluru-air-pollution-eda.